# Unit 03 · Baselines: the visual hull and the memorisation floor

Before any neural network: two ways to get a 3D pelvis from the two X-rays that need **no training at all**
(the visual hull) or **only a lookup** (nearest neighbour). Every later number (the network in unit 05, the
experiment in unit 06) is read against this table: a network that doesn't beat these hasn't learned anything.

**Needs:** units 01–02 moved into `src/xray3d` (`geometry.project_pinhole`, `shape.mesh_from_occupancy`,
`metrics.dice`, `metrics.evaluate`) and the processed dataset v1 (built in unit 02, `build_dataset.ipynb`).

| Part | After video | What you build | Moves into | Suggested driver |
|---|---|---|---|---|
| A | 07 · From two views back to 3D | voxel → pixel, back-projection, the visual hull | `baselines.py` | Noa (lead) |
| B | 08 · The memorisation floor | image features, nearest neighbour, the oracle and the mean shape; the results table | `baselines.py` | Adir (lead) |

Within a part, swap the keyboard on every TODO: one types, the other explains. Record it in `LOG.md`.

In [ ]:
# --- setup (written by Claude, no need to change it): the same cell works locally, on Colab and on Kaggle ---
import os, sys, subprocess
from pathlib import Path

UNIT = "03-baselines"
REPO_URL = "https://github.com/AdirBuskila/pelvis-xray-3d"
IN_CLOUD = "google.colab" in sys.modules or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_CLOUD:
    root = Path("/content" if "google.colab" in sys.modules else "/kaggle/working") / "pelvis-xray-3d"
    if not root.exists():  # Kaggle: turn Internet on in the notebook settings first
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
else:  # local: the repo is the first folder above this notebook that has a pyproject.toml
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root / "units" / UNIT)
for p in (root / "src", Path.cwd()):  # import xray3d (even before `pip install -e .`) and this unit's checks
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import xray3d
from xray3d import data, env, plotting
import test_unit as T

env.summary()

In [ ]:
from xray3d import dataset, geometry, metrics, shape, prep
need = [(geometry, "project_pinhole"), (shape, "mesh_from_occupancy"), (metrics, "dice"), (metrics, "evaluate")]
missing = [f"{m.__name__}.{f}" for m, f in need if not hasattr(m, f)]
print("unit 01-02 functions missing from src/xray3d:", missing or "none, good")
dataset.ensure_local()                       # Colab: unzips the dataset from Drive; Kaggle: attach pelvis-xray3d-v1

G = dataset.geometry()
N_PX, N_VOX = G["detector_px"], G["cube_n"]
CAMS = {v: dataset.camera(v) for v in dataset.VIEWS}          # K, R, t of each view, in the cube frame (mm)
SPLIT = {k: dataset.subjects(k) for k in ("train", "val", "test")}
print({k: len(v) for k, v in SPLIT.items()}, "subjects in train / val / test")
KEYS = ("occ", "ap", "lat", "ap_bone", "lat_bone")
DATA = {k: dataset.load_arrays(SPLIT[k], KEYS) for k in SPLIT}      # DATA["test"]["occ"] is (n_test, 64, 64, 64)
SHOW = SPLIT["test"][0]                                             # the subject the pictures use
d = dataset.load_subject(SHOW)
plotting.show_row([d["ap"], d["lat"], d["ap_bone"], d["lat_bone"]],
                  [f"{SHOW} ap (input)", "lat (input)", "ap_bone (oracle)", "lat_bone (oracle)"], size=3.8)

The first two images are what a real X-ray looks like (here simulated from the whole CT). The `*_bone` images
are the same rendering of **the pelvis alone**: from them a perfect silhouette is free. In a hospital nobody gives
you that: you'd have to segment the pelvis in the X-ray first. Keep that in mind for part A.

Helpers for the pictures (Claude's): an **axial** layer of a cube, seen from the feet like a CT slice
(anterior at the top, patient's Right on the left of the image), and the coronal one from unit 02.

In [ ]:
def axial(vol3d, k):
    """Axial layer k of a cube (RAS axes): rows Anterior (top) -> Posterior, columns patient's Right -> Left."""
    return np.asarray(vol3d)[:, :, k].T[::-1, ::-1]

gt = d["occ"].astype(bool)
ks = np.flatnonzero(gt.any(axis=(0, 1)))                    # the layers (k) that hold some pelvis
LEVELS = {"low (hip sockets)": int(ks[0] + 0.30 * (ks[-1] - ks[0])),
          "middle (inside the ring)": int(ks[0] + 0.45 * (ks[-1] - ks[0])),
          "high (sacrum, iliac wings)": int(ks[0] + 0.65 * (ks[-1] - ks[0]))}
print("pelvis spans layers", ks[0], "to", ks[-1], "| picture levels", LEVELS)

---
## Part A · From two views back to 3D (after video 07)

Video 07: an X-ray squashes 3D onto 2D along rays. Going back, every pixel only says "something lies
**somewhere** on my ray". A voxel is on a ray if it **projects** onto that pixel, so everything in this part
starts from one question: *for each voxel centre, which pixel of view v does it land on?* `dataset.camera(v)`
and your unit-01 `project_pinhole` answer the continuous part; the rest is yours.

### TODO A1 · `pixel_index`  ·  suggested driver: **Noa**

For many 3D points (mm, cube frame) and one camera `(K, R, t)`: the **pixel index** (row, column) each point
lands on, and whether it lands **on the detector** at all. `project_pinhole` returns `(u, v)` = (column, row)
with pixel centres at whole numbers (the image centre is 63.5). Points can come as any array `(..., 3)`; give
back `row`, `col`, `inside` with the points' shape without the last axis.

> **Nudge:** pixel c covers u from c − 0.5 to c + 0.5. Which numpy function takes u to the index of the pixel it falls in (int() is not it)? A point is on the detector only if both indices are in 0 … size − 1. Flatten the points to (N, 3) for project_pinhole and reshape at the end.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
from xray3d import geometry  # the module, not the name: call geometry.project_pinhole(...)

def pixel_index(points_mm, camera, size=128):
    """Which pixel of a view does each point land on?

    points_mm : (..., 3) points in the cube frame (mm)
    camera    : (K, R, t) of the view, from dataset.camera(view)
    size      : the detector is size x size pixels
    returns   : row, col (int arrays, shape (...)) and inside (bool, shape (...)): True where the point lands
                on the detector. row/col are only meaningful where inside is True.
    """
    raise NotImplementedError("TODO A1")

In [ ]:
T.check_pixel_index(pixel_index)

In [ ]:
centres = dataset.voxel_centers_mm()                        # (64, 64, 64, 3), the cube frame
for v in dataset.VIEWS:
    row, col, inside = pixel_index(centres, CAMS[v], N_PX)
    print(f"{v:<3}: {inside.mean():6.1%} of the cube's voxel centres land on the detector;"
          f" pelvis voxels of {SHOW} on it: {inside[gt].mean():.1%};"
          f" columns used by the pelvis {col[gt].min()}..{col[gt].max()}, rows {row[gt].min()}..{row[gt].max()}")

**Explain-back:** not every voxel of the 320 mm cube is on the AP detector. Which ones fall off, and why the
ones at the **front** (anterior) rather than the back? (Magnification = source–detector / source–point. The
front of the cube is 750 − 160 = 590 mm from the source.) Why is it fine for this project?

### TODO A2 · `backproject`  ·  suggested driver: **Adir**

**Back-projection:** the opposite of a projection. Every voxel takes the value of the pixel it lands on,
so the image is **smeared back along its rays** (a voxel off the detector gets 0: no ray of this image passes
through it). With no `points_mm` given, use every voxel centre of the cube (`dataset.voxel_centers_mm()`).

> **Nudge:** your A1 already tells you, for each point, the pixel and whether it exists. Start from an array of zeros with the points' shape and fill only the inside ones, by fancy indexing image[row, col].

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def backproject(image, camera, points_mm=None):
    """Smear one view's image back into 3D along its rays.

    image     : (size, size) one view's image (any dtype: a DRR, or a bool silhouette)
    camera    : (K, R, t) of that view
    points_mm : (..., 3) where to evaluate; default: every voxel centre of the cube, (64, 64, 64, 3)
    returns   : (...) the value of the pixel each point lands on, 0 where it lands off the detector
    """
    raise NotImplementedError("TODO A2")

In [ ]:
T.check_backproject(backproject)

In [ ]:
bp = {v: backproject(d[f"{v}_bone"], CAMS[v]) for v in dataset.VIEWS}
k = LEVELS["middle (inside the ring)"]
fig, ax = plt.subplots(1, 4, figsize=(17, 4.4))
for a, (img, title) in zip(ax, [(axial(bp["ap"], k), "AP smeared back"), (axial(bp["lat"], k), "LAT smeared back"),
                                (axial(bp["ap"] * bp["lat"], k), "AP x LAT: where both see bone"),
                                (axial(gt, k), "the true pelvis")]):
    a.imshow(img, cmap="bone", interpolation="nearest"); a.set_title(f"{title}\naxial layer k = {k}"); a.axis("off")
plt.tight_layout(); plt.show()

One view is a set of stripes: the AP image is constant along the beam (front to back), the LAT image along
left–right. Where the stripes cross is where *both* views saw bone. That crossing, made sharp with
silhouettes, is the **visual hull**.

### TODO A3 · `visual_hull`  ·  suggested driver: **Noa**

The **visual hull**: keep a voxel only if it lands **inside the silhouette in every view**. A silhouette is a
bool image (True = the bone is somewhere on this ray). Views come as dicts keyed by name (`"ap"`, `"lat"`),
so the same function works with one view, two, or ten. Return `uint8` 0/1 like the dataset's occupancy.

> **Nudge:** a voxel survives one view if the silhouette is True at its pixel: that's your A2 on a bool image. 'Every view' is an AND over the views. Start from 'everything survives' (or from the first view).

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def visual_hull(silhouettes, cameras, points_mm=None):
    """Carve the voxels that every silhouette sees.

    silhouettes : {view: (size, size) bool}   e.g. {"ap": d["ap_bone"] > 1e-3, "lat": ...}
    cameras     : {view: (K, R, t)}          (views missing from silhouettes are ignored)
    points_mm   : (..., 3) voxel centres; default: the cube's (64, 64, 64, 3)
    returns     : (...) uint8, 1 = inside every silhouette
    """
    raise NotImplementedError("TODO A3")

In [ ]:
T.check_visual_hull(visual_hull)

In [ ]:
THRESH = 1e-3                                   # the same silhouette threshold as unit 02's geometry check
def silhouettes(sub, i, thresh=THRESH):
    return {v: DATA[sub][f"{v}_bone"][i] > thresh for v in dataset.VIEWS}

hull = visual_hull(silhouettes("test", 0), CAMS).astype(bool)
print(f"{SHOW}: true pelvis {gt.sum()} voxels, hull {hull.sum()} ({hull.sum() / gt.sum():.1f} x as many)")
print(f"  pelvis voxels inside the hull: {(gt & hull).sum() / gt.sum():.1%}   Dice {metrics.dice(gt, hull):.3f}")
fig, ax = plt.subplots(1, 4, figsize=(17, 4.6))
for a, (name, kk) in zip(ax, LEVELS.items()):
    a.imshow(prep.overlap_image(axial(gt, kk), axial(hull, kk)), interpolation="nearest"); a.set_title(f"axial, {name}"); a.axis("off")
ax[3].imshow(prep.overlap_image(prep.coronal(gt), prep.coronal(hull)), interpolation="nearest"); ax[3].set_title("coronal"); ax[3].axis("off")
fig.suptitle("true pelvis vs visual hull: white = both, red = hull only (FP), blue = pelvis only (FN)")
plt.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

In [ ]:
fig = plt.figure(figsize=(13, 6))
for c, (m, title) in enumerate([(gt, f"{SHOW}: the true pelvis"), (hull, "its visual hull (AP + LAT)")]):
    v_, f_ = shape.mesh_from_occupancy(m)
    prep.show_mesh(v_, f_, title, ax=fig.add_subplot(1, 2, c + 1, projection="3d"), view=(25, 60))
plt.show()

**Explain-back:**
1. The red in the middle axial layer is the hollow **inside the pelvic ring** (the true pelvis, below the
   pelvic inlet), and in the high one the bowl between the iliac wings. Which AP ray and which LAT ray pass
   through a voxel in the middle of the hollow, and what do they hit on their way? Why can no number of views
   **around the head–feet axis** carve it out? (Video 07: concavities are invisible to silhouettes; only a view
   looking *down* through the inlet would help, and even that one sees the sacrum behind it.)
2. Almost every pelvis voxel is inside the hull, but not 100 %. Where do the few blue voxels come from?
   (A voxel is kept or carved by **its centre's** pixel alone.)
3. The silhouettes came from `*_bone`, the pelvis rendered alone. Why does that make this hull an **upper
   bound** for any silhouette method on real X-rays? What would you have to build first to use it on a real
   AP image?

**The threshold.** `*_bone > 1e-3` keeps every ray that grazes bone. A bigger threshold drops the rays that
cross only a thin edge. Here's the trade-off over the **validation** subjects (never tune on test):

In [ ]:
ths = [1e-3, 0.02, 0.05, 0.1, 0.2, 0.4, 0.8]
res = []
for th in ths:
    dd, cv = [], []
    for i in range(len(SPLIT["val"])):
        g_ = DATA["val"]["occ"][i].astype(bool)
        h_ = visual_hull(silhouettes("val", i, th), CAMS).astype(bool)
        dd.append(metrics.dice(g_, h_)); cv.append((g_ & h_).sum() / g_.sum())
    res.append((np.mean(dd), np.mean(cv)))
res = np.array(res)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ths, res[:, 0], "o-", color="#3b6fb6"); ax[0].set_title("hull Dice (val mean)")
ax[1].plot(ths, res[:, 1], "o-", color="#3b6fb6"); ax[1].set_title("fraction of the pelvis inside the hull (val mean)")
for a in ax:
    a.set_xscale("log"); a.set_xlabel("silhouette threshold on *_bone (line integral)"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
for th, (dv, cv) in zip(ths, res):
    print(f"threshold {th:<6} Dice {dv:.3f}   pelvis inside the hull {cv:.1%}")

**Answer together:** the Dice goes up with the threshold, and the hull stops containing the whole pelvis.
Which number matters more for a baseline that claims to be "what the silhouettes allow"? We keep 1e-3 for the
results table (the hull stays a **superset** of the pelvis, which is the definition of a visual hull).

**Part A done.** Move `pixel_index`, `backproject` and `visual_hull` into `src/xray3d/baselines.py` (with
`from xray3d import dataset, geometry` at the top: import the *module* and call `geometry.project_pinhole`, so
a function that hasn't been moved yet fails only when it's called, not every test in the file).

---
## Part B · The memorisation floor (after video 08)

The cheapest "reconstruction": find the **training** patient whose X-rays look most like the test patient's,
and answer with **that patient's pelvis**. No 3D reasoning at all. Tatarchenko et al. (2019) showed that
single-view reconstruction networks often score about the same as this kind of retrieval: they recognise
and recall, rather than reconstruct. So this is the floor a network must clear before we believe it learned
geometry.

The inputs are the realistic ones now: `ap` and `lat`, the whole-body DRRs, not the oracle `*_bone` images.

### TODO B1 · `nn_features`  ·  suggested driver: **Adir**

Turn a subject's two X-rays into **one vector** to compare. Two steps per image, in this order:
**downsample** to `size × size` by averaging blocks (128 / size pixels square), then **standardise** the small
image (subtract its mean, divide by its std; a constant image becomes all zeros). Flatten each (row by row),
AP first, then LAT. `lat=None` means AP only. Accept one subject `(128, 128)` or a batch `(N, 128, 128)`.

> **Nudge:** a block mean is a reshape: (128, 128) → (size, 128 // size, size, 128 // size), then mean over the two block axes. Standardising each image on its own makes brightness and contrast drop out: why might two patients' DRRs differ in brightness although their pelvises are alike? For a batch, do the same with one more axis in front.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def nn_features(ap, lat=None, size=32):
    """Feature vector(s) for nearest-neighbour retrieval.

    ap, lat : (H, W) or (N, H, W) images (lat may be None: AP only); H and W divisible by size
    size    : side of the downsampled image
    returns : (V * size * size,) for one subject or (N, V * size * size) for a batch, V = 1 or 2 views;
              each view's part has mean 0 and std 1 (or is all zeros for a flat image)
    """
    raise NotImplementedError("TODO B1")

In [ ]:
T.check_nn_features(nn_features)

In [ ]:
SIZE = 32
f = nn_features(d["ap"], d["lat"], SIZE)
plotting.show_row([d["ap"], f[:SIZE * SIZE].reshape(SIZE, SIZE), f[SIZE * SIZE:].reshape(SIZE, SIZE)],
                  ["ap, 128 x 128", f"AP feature ({SIZE} x {SIZE}, standardised)", "LAT feature"], size=3.8)
print(f"{f.size} numbers per subject")

### TODO B2 · `nearest_neighbour`  ·  suggested driver: **Noa**

For every query vector: the index of the **closest training vector** in Euclidean (L2) distance.

> **Nudge:** you need a (Q, N) table of distances. Broadcasting (Q, 1, D) against (1, N, D) builds all differences at once; which axis do you sum over, and which axis does argmin run along? (The square root doesn't change which one is smallest.)

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def nearest_neighbour(train_feats, query_feats):
    """Index of the nearest training row for every query row (L2 distance).

    train_feats : (N, D);  query_feats : (Q, D)
    returns     : (Q,) int, values in 0 .. N - 1
    """
    raise NotImplementedError("TODO B2")

In [ ]:
T.check_nearest_neighbour(nearest_neighbour)

**Choosing `size` (on validation, not test).** Smaller images ignore small shifts and noise but also small
differences in shape; bigger ones compare more pixels. Train → val, mean Dice of the retrieved pelvis:

In [ ]:
def feats(sub, size, views=("ap", "lat")):
    return nn_features(DATA[sub]["ap"], DATA[sub]["lat"] if "lat" in views else None, size)

for size in (4, 8, 16, 32, 64):
    j = nearest_neighbour(feats("train", size), feats("val", size))
    dv = [metrics.dice(DATA["val"]["occ"][q], DATA["train"]["occ"][jj]) for q, jj in enumerate(j)]
    print(f"size {size:>2}: val Dice {np.mean(dv):.3f}   retrieved {[SPLIT['train'][jj] for jj in j]}")

With only a few validation subjects these numbers move by chance; we keep `SIZE = 32` unless val disagrees
clearly. Now the test subject of the pictures: its X-rays, the X-rays of the training patient it retrieved,
and the two pelvises on top of each other.

In [ ]:
NN = nearest_neighbour(feats("train", SIZE), feats("test", SIZE))
j = int(NN[0])
retrieved = DATA["train"]["occ"][j].astype(bool)
fig, ax = plt.subplots(1, 5, figsize=(19, 4.2))
for a, (img, title) in zip(ax, [(d["ap"], f"query {SHOW} ap"), (DATA["train"]["ap"][j], f"retrieved {SPLIT['train'][j]} ap"),
                                (d["lat"], "query lat"), (DATA["train"]["lat"][j], "retrieved lat")]):
    a.imshow(img, cmap="gray"); a.set_title(title); a.axis("off")
ax[4].imshow(prep.overlap_image(prep.coronal(gt), prep.coronal(retrieved)), interpolation="nearest")
ax[4].set_title(f"pelvis: Dice {metrics.dice(gt, retrieved):.3f}"); ax[4].axis("off")
plt.tight_layout(); plt.show()

### TODO B3 · `oracle_neighbour`  ·  suggested driver: **Adir**

How good could retrieval **ever** be with this training set? The **oracle** cheats: it looks at the test
pelvis (the ground truth!) and picks the training pelvis with the **highest Dice** against it. No method that
answers with a training shape can beat it, so it's the ceiling of retrieval, not a method.

> **Nudge:** one Dice per training shape (your unit-02 metrics.dice), then the index of the best. Why isn't 'the training shape that covers most of gt' the same thing?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
from xray3d import metrics  # the module, not the name: call metrics.dice(...)

def oracle_neighbour(train_occ, gt_occ):
    """Index of the training shape with the highest Dice against gt (uses the ground truth: an upper bound).

    train_occ : (N, n, n, n) 0/1;  gt_occ : (n, n, n) 0/1
    returns   : int
    """
    raise NotImplementedError("TODO B3")

In [ ]:
T.check_oracle_neighbour(oracle_neighbour)

### TODO B4 · `mean_shape`  ·  suggested driver: **Noa**

The laziest baseline of all: answer **the same shape for everyone**, the training pelvises' **voxel-wise
mean**, thresholded at `level` (≥ level: a voxel is in if at least half the training pelvises have it).
It works only because every pelvis sits in the cube the same way (unit 02: the box centre, RAS axes, real mm).

> **Nudge:** the mean of 0/1 values along the subjects' axis is the fraction of subjects that have the voxel. Then the same threshold rule as unit 02's occupancy.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def mean_shape(train_occ, level=0.5):
    """The training shapes' voxel-wise mean, thresholded: (n, n, n) uint8, 1 where mean >= level."""
    raise NotImplementedError("TODO B4")

In [ ]:
T.check_mean_shape(mean_shape)

---
## The results table

Every method, every test subject, every metric of your `metrics.evaluate`. One row per (method, subject), so
unit 06 can pair them subject by subject. The file is `results/baselines.csv` (small; commit it).

| method | sees | uses training data | uses the test GT |
|---|---|---|---|
| `hull_oracle_sil` | the pelvis-only silhouettes (`*_bone`) | no | no, but oracle inputs |
| `nn_ap_lat` | `ap` + `lat` | retrieval | no |
| `nn_ap` | `ap` only | retrieval | no |
| `mean_shape` | nothing | the mean | no |
| `oracle_retrieval` | the test pelvis | retrieval | **yes** (a ceiling, not a method) |

In [ ]:
import csv, time
t0 = time.time()
tr_occ = DATA["train"]["occ"]
NN1 = nearest_neighbour(feats("train", SIZE, ("ap",)), feats("test", SIZE, ("ap",)))
MEAN = mean_shape(tr_occ)
rows = []
for q, sid in enumerate(SPLIT["test"]):
    g_ = DATA["test"]["occ"][q]
    o = oracle_neighbour(tr_occ, g_)
    preds = {"hull_oracle_sil": (visual_hull(silhouettes("test", q), CAMS), ""),
             "nn_ap_lat": (tr_occ[NN[q]], SPLIT["train"][NN[q]]),
             "nn_ap": (tr_occ[NN1[q]], SPLIT["train"][NN1[q]]),
             "mean_shape": (MEAN, ""),
             "oracle_retrieval": (tr_occ[o], SPLIT["train"][o])}
    for method, (p, src) in preds.items():
        rows.append({"method": method, "subject": sid, "retrieved": src, **metrics.evaluate(g_, p)})
out = env.results_root() / "baselines.csv"
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w", newline="") as fo:
    w = csv.DictWriter(fo, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows({k: (f"{v:.4f}" if isinstance(v, float) else v) for k, v in r.items()} for r in rows)
print(f"wrote {out} ({len(rows)} rows, {time.time() - t0:.0f} s); {len(SPLIT['train'])} training subjects, {len(SPLIT['test'])} test")

METHODS = list(dict.fromkeys(r["method"] for r in rows))
COLS = ["dice", "iou", "hd95", "assd", "nsd_1.5mm", "nsd_5mm", "chamfer"]
print(f"\n{'method':<18}" + "".join(f"{c:>15}" for c in COLS))
for m in METHODS:
    vals = np.array([[r[c] for c in COLS] for r in rows if r["method"] == m])
    print(f"{m:<18}" + "".join(f"{mu:8.3f} ±{sd:5.2f}" for mu, sd in zip(vals.mean(0), vals.std(0))))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for a, c, label in [(ax[0], "dice", "Dice (higher is better)"), (ax[1], "hd95", "HD95, mm (lower is better)")]:
    for y, m in enumerate(METHODS):
        v = np.array([r[c] for r in rows if r["method"] == m])
        a.scatter(v, np.full(len(v), y) + np.linspace(-0.15, 0.15, len(v)), s=22, color="#3b6fb6", alpha=0.7)
        a.plot([v.mean()] * 2, [y - 0.3, y + 0.3], color="#1d1d1d", lw=2)
    a.set_yticks(range(len(METHODS)), METHODS); a.invert_yaxis(); a.set_xlabel(label); a.grid(axis="x", alpha=0.3)
fig.suptitle("baselines on the test subjects: one dot per subject, the bar is the mean")
plt.tight_layout(); plt.show()
same = sum(r["retrieved"] == o["retrieved"] for r, o in zip([r for r in rows if r["method"] == "nn_ap_lat"],
                                                              [r for r in rows if r["method"] == "oracle_retrieval"]))
print(f"nearest neighbour picked the oracle's shape for {same} of {len(SPLIT['test'])} test subjects")

**Explain-back:**
1. Order the rows by Dice and by HD95. Where does the hull land, and why is a *superset* of the pelvis bad for
   Dice but not as bad for "pelvis inside the hull"? Which metric would a surgeon planning screws care about?
2. The **mean shape**, the same answer for everyone, is as good as nearest neighbour or better. What does that
   say about how much the X-rays told the retrieval? (Think of how many training pelvises it can choose from,
   and how often it found the oracle's. Look at the query and retrieved X-rays above: what fills most of the
   pixels, the pelvis or the body's outline and soft tissue? Which of the two does an L2 distance mostly compare?) What should happen to `nn_ap_lat` with the ~300 subjects of the full
   dataset, and to `mean_shape`?
3. Tatarchenko et al. call retrieval and clustering the baselines that single-view networks fail to beat by much.
   Why is that a **floor** for our network and the oracle a **ceiling** for retrieval? If the network in unit 05
   scores between the two, what has it shown and what hasn't it?
4. `nn_ap` vs `nn_ap_lat`: with 9 test subjects, is the difference real? (Unit 06 does this properly: seeds,
   paired differences.) The class deck showed hull ≈ 0.38 and NN ≈ 0.57 Dice on a different setup (parallel
   rays, the 27 subjects leave-one-out, centroid-centred cubes): what changed?

---
## Wrap-up: run the checks against `src/xray3d`

Functions you haven't moved yet show up as **skipped**, moved ones as **passed**.

In [ ]:
!{sys.executable} -m pytest -q -rs .

Fill in `LOG.md`: driver per TODO, hints used, the explain-back answers and the results table. Then ask Claude
to review the code like a TA, and to commit (with `results/baselines.csv`).